<a href="https://colab.research.google.com/github/kev841/test-1/blob/main/notebook/diabetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Relevant Libraries

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score


from sklearn.metrics import confusion_matrix as sk_confusion_matrix # Aliased import
from sklearn.metrics import classification_report

# Load Data

In [ ]:
data = pd.read_csv('/content/diabetes.csv')
data.head(20)

In [ ]:
data.describe(include='all')

In [ ]:
data.nsmallest(5, 'Glucose')

check for missing data


In [ ]:
data.isnull().sum()

In [ ]:
# Plotting every column in its own separate graph
data.plot(subplots=True, layout=(3, 3), figsize=(15, 10), title="Dataset Overview")
plt.tight_layout()
plt.show()


In the dataset there are multiple records of zero entries which need to be handled

In [ ]:
zero_cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
for c in zero_cols:
  print(c, "zero count:", (data[c]==0).sum())

hanndilng zeros glucose column

In [ ]:
# 1. Replace 0 with NaN so they aren't included in median calculation
data['Glucose'] = data['Glucose'].replace(0, np.nan)

# 2/ Fill NaN values with the median of their respective 'Outcome' group
data['Glucose'] = data['Glucose'].fillna(data.groupby('Outcome')['Glucose'].transform('median'))

# 3. Verify there are no zeros or NaNs left
print("Remaining Zeros:", (data['Glucose'] == 0).sum())
print("Remaining NaNs:", data['Glucose'].isnull().sum())

handling the blood pressure and bmi column

In [ ]:
data[['BloodPressure','BMI']] = data[['BloodPressure','BMI']].replace(0, np.nan)

fill with grouped median

In [ ]:
data['BloodPressure'] = data['BloodPressure'].fillna(data.groupby('Outcome')['BloodPressure'].transform('median'))
data['BMI'] = data['BMI'].fillna(data.groupby('Outcome')['BMI'].transform('median'))
data[['BloodPressure','BMI']].isnull().sum()

handled the skin thickness nad insulin columns

In [ ]:
data['SkinThickness'] = data['SkinThickness'].replace(0, np.nan)
data['SkinThickness']= data['SkinThickness'].fillna(data.groupby('Outcome')['SkinThickness'].transform('median'))
data['SkinThickness'].isnull().sum()
data['Insulin'] = data['Insulin'].replace(0, np.nan)
data['Insulin']= data['Insulin'].fillna(data.groupby('Outcome')['Insulin'].transform('median'))
data['Insulin'].isnull().sum()

In [ ]:
data.describe(include='all')

In [ ]:
plt.figure(figsize=(8,8))
plt.title("Glucose against Outcome")
sns.violinplot(x=data['Outcome'], y=data['Glucose'], data=data, inner="box")
plt.show()

correlation analysis to identify which factors influence the 'Outcome'

In [ ]:
correlation_matrix = data.corr()
print(data.corr()['Outcome'].sort_values(ascending=False))

In [ ]:
# Visualizing the correlation using a heatmap
sns.heatmap(correlation_matrix, annot=True)
plt.show()

i am going to be using Random Forest to measure how much each variable contributes to predicting the Outcome of a person having diabetes

## Prepare Data

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = data[['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']]
y = data['Outcome']

# train model
model = RandomForestClassifier()
model.fit(X,y)

importance = model.feature_importances_
importance

In [ ]:
# create a table to show each feature importance
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print(feature_importance)

# Plot
plt.bar(feature_importance["Feature"], feature_importance["Importance"])
plt.xticks(rotation=45)
plt.title("Feature Importance for Diabetes Prediction")
plt.show()

# Train model using a Logistic Regression

In [ ]:
# Split Data
X = data[['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']]
Y = data['Outcome']

In [ ]:
#train test and split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

from sklearn.preprocessing import StandardScaler

# 1. Initialize the scaler
scaler = StandardScaler()

# 2. Fit and transform the TRAINING data
X_train_scaled = scaler.fit_transform(X_train)

# 3. Transform the TEST data
X_test_scaled = scaler.transform(X_test)

from sklearn.ensemble import RandomForestClassifier

# limit 'max_depth' so the trees don't grow forever and memorize noise
# increase 'min_samples_leaf' so a pattern must apply to at least 5 people to be counted
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,            # Prevents trees from getting too complex
    min_samples_leaf=5,      # Ensures the model finds general patterns, not unique cases
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train_scaled, Y_train)
print(f"New Training Acc: {rf_model.score(X_train_scaled, Y_train):.4f}")
print(f"New Test Acc: {rf_model.score(X_test_scaled, Y_test):.4f}")


# 3. Predict labels (0 or 1)
Y_pred_rf = rf_model.predict(X_test_scaled)

# 4. Predict probabilities (for that sweet AUC)
Y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# 5. Check the new scores
print(f"RF Accuracy: {accuracy_score(Y_test, Y_pred_rf):.4f}")
print(f"RF Recall: {recall_score(Y_test, Y_pred_rf):.4f}")
print(f"RF AUC: {roc_auc_score(Y_test, Y_prob_rf):.4f}")


# confusion Matrix

In [ ]:
cm = sk_confusion_matrix(Y_test, Y_pred)
print("Confusion Matrix:", cm)

In [ ]:
#classification report
print(classification_report(Y_test, Y_pred))

In [ ]:
# Check training accuracy
train_acc = rf_model.score(X_train_scaled, Y_train)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {accuracy_score(Y_test, Y_pred_rf):.4f}")


In [52]:
!jupyter nbconvert --clear-output --inplace /content/diabetes.ipynb

[NbConvertApp] WARNING | pattern '/content/diabetes.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execu